# 03 - Rotulagem baseada em regras

Criação dos rótulos `ciclo`, `fase_enso` e `foco_critico` na base limpa e atualização das tabelas mensais integradas.

## 1. Leitura da base limpa

In [1]:
import pandas as pd

df = pd.read_parquet("../dados/queimadas_limpo.parquet")
print("Dimensões carregadas:", df.shape)

Dimensões carregadas: (1633615, 18)


## 2. Aplicação das regras de rotulagem

Classificamos `fase_enso` em `El Niño` (`roni >= 0.5`), `La Niña` (`roni <= -0.5`), `Neutro` no intervalo intermediário e `Sem dado RONI` quando ausente. Em `foco_critico`, atribuímos `1` quando `RiscoFogo >= 0.8` e `DiaSemChuva >= 15`.

In [2]:
def rotular_ciclo(ano_mes):
    if "2024-12" <= ano_mes <= "2025-11":
        return "2024-2025 (Sem El Niño)"
    return "2025-2026 (El Niño)"

def rotular_enso(roni):
    if pd.isna(roni):
        return "Sem dado RONI"
    elif roni >= 0.5:
        return "El Niño"
    elif roni <= -0.5:
        return "La Niña"
    else:
        return "Neutro"

df["ciclo"] = df["ano_mes"].apply(rotular_ciclo)
df["fase_enso"] = df["roni"].apply(rotular_enso)
df["foco_critico"] = ((df["RiscoFogo"] >= 0.8) & (df["DiaSemChuva"] >= 15)).astype(int)

display(df["ciclo"].value_counts())
display(df["fase_enso"].value_counts())
display(df["foco_critico"].value_counts())

ciclo
2024-2025 (Sem El Niño)    853402
2025-2026 (El Niño)        780213
Name: count, dtype: int64

fase_enso
La Niña          808507
El Niño          373736
Sem dado RONI    293864
Neutro           157508
Name: count, dtype: int64

foco_critico
0    1145676
1     487939
Name: count, dtype: int64

## 3. Salvamento da base rotulada e das tabelas mensais

In [3]:
df.to_parquet("../dados/queimadas_rotulado.parquet", index=False)

for nome in ["integrado_mensal_brasil.csv", "integrado_mensal_biomas.csv", "integrado_mensal_estados.csv"]:
    caminho = f"../dados/{nome}"
    tabela = pd.read_csv(caminho)
    tabela["ciclo"] = tabela["ano_mes"].apply(rotular_ciclo)
    tabela["fase_enso"] = tabela["roni"].apply(rotular_enso)
    tabela.to_csv(caminho, index=False)

print("Arquivos atualizados em ../dados/")

Arquivos atualizados em ../dados/
